# Interpretability of a trained model

For the proposed model of one dataset and split:

* the fingerprint of every HMM state: share of the training packets that the Viterbi path assigns to it, median payload, median and 99th-percentile IAT of these packets, long-run share, expected dwell in packets and seconds, and the effective degrees of freedom of the MDN (mixture-weighted mean of nu at the geometric mean flow length);
* the response of every state to the flow length: generated median payload, 90th-percentile IAT and P(IAT > 1 s) for flows of 5, 50, 500 and 5,000 packets.

Writes `state_fingerprint.csv` and `flow_length_response.csv` into `OUT_DIR/artifacts/<SOURCE_MODEL_KEY>/`; `scripts/summarize_interpretability.py` compares the splits.

In [ ]:
SEED             = 0
DATASET          = 'facebook_video'
RAW_PATH_STR     = 'data/facebook_video.csv'
OUT_DIR          = 'runs/facebook_video/seed0'
SOURCE_MODEL_KEY = 'proposed'

In [ ]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import load_packets, read_split
from hmm_mdn import GlobalStateLenMDN, sample_mixture

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
warnings.filterwarnings('ignore', message='X does not have valid feature names')

In [ ]:
df = load_packets(RAW_PATH_STR)
train_flows, test_flows = read_split(Path(OUT_DIR) / 'split.json')

art = Path(OUT_DIR) / 'artifacts' / SOURCE_MODEL_KEY
meta = json.loads((art / 'meta.json').read_text())
hmm_boot, scaler_pkt, scaler_len = (joblib.load(art / f) for f in ('hmm.pkl', 'scaler_pkt.pkl', 'scaler_len.pkl'))
K_STATES = meta['K']
IDLE_K = K_STATES - 1 if meta['idle'] else -1
DIST_FAMILY = meta['dist_family']
TEMPERATURE_IDLE = meta.get('temperature_idle', 1.0)
model_mdn = GlobalStateLenMDN(K_STATES, hidden=meta['H'], n_mix=meta['M'], use_flow_len=not meta['no_flow_len']).to(device)
model_mdn.load_state_dict(torch.load(art / 'mdn.pth', map_location=device, weights_only=True))
model_mdn.eval()

In [ ]:
# State fingerprint
K, A = hmm_boot.n_components, hmm_boot.transmat_
# long-run share of each state for a chain started from the start distribution (power iteration;
# an eigenvector of A is not unique when a state without posterior mass is absorbing)
occ = hmm_boot.startprob_.copy()
for _ in range(100_000):
    nxt = occ @ A
    if np.abs(nxt - occ).max() < 1e-13:
        break
    occ = nxt
diag = np.diag(A)
dwell_pkts = np.where(diag < 1.0, 1.0 / np.clip(1.0 - diag, 1e-12, None), np.inf)   # E[D_k] = 1 / (1 - a_kk)

tr = df[df['flow_id'].isin(train_flows)].reset_index(drop=True)
Z = scaler_pkt.transform(np.column_stack([np.log1p(tr['payload_length']), np.log1p(tr['time_diff'])]))
state = np.empty(len(tr), dtype=int)
for _, idx in tr.groupby('flow_id', sort=False).indices.items():
    state[idx] = hmm_boot.predict(Z[idx])                                 # Viterbi path of each flow
tr['state'] = state
fp = (tr.groupby('state')
        .agg(share=('payload_length', 'size'), payload_med=('payload_length', 'median'),
             iat_med_ms=('time_diff', lambda s: 1e3 * s.median()),
             iat_p99_s=('time_diff', lambda s: s.quantile(0.99)),
             iat_mean_s=('time_diff', 'mean'))
        .reindex(range(K)))
fp['share'] /= len(tr)
fp['long_run_share'], fp['dwell_pkts'] = occ, dwell_pkts
fp['dwell_s'] = fp['dwell_pkts'] * fp['iat_mean_s']
with torch.no_grad():                                                     # nu_eff at xi = 0
    x = torch.zeros(K, K + 1, device=device)
    x[:, :K] = torch.eye(K, device=device)
    w_c, mu_c, sg_c, nu_c = model_mdn(x)
fp['nu_eff'] = (w_c * nu_c.squeeze(-1)).sum(-1).cpu().numpy()
unused = fp['share'].isna()
fp.loc[unused, ['long_run_share', 'dwell_pkts', 'dwell_s', 'nu_eff']] = np.nan
fp.index.name = 'state'
print(f'idle-initialized state: {IDLE_K if IDLE_K >= 0 else None} | states without training packets: '
      f'{list(fp.index[unused])}')
print(fp.round(3))
fp.to_csv(art / 'state_fingerprint.csv')

# Response of every state to the flow length, sampled as in generation
used = [k for k in range(K) if not unused.iloc[k]]


def conditional_summary(L, n=20000):
    lz, rows = float(scaler_len.transform([[np.log1p(L)]])[0, 0]), []
    with torch.no_grad(), np.errstate(over='ignore'):
        for k in used:
            x = torch.zeros(n, K + 1, device=device)
            x[:, k] = 1.0
            x[:, K] = lz
            s = sample_mixture(*model_mdn(x), torch.full((n,), k, device=device), IDLE_K, TEMPERATURE_IDLE, DIST_FAMILY)
            r = scaler_pkt.inverse_transform(s.cpu().numpy())
            iat = np.clip(np.expm1(r[:, 1]), 1e-6, 3600.0)        # IAT range of the generator
            pay = np.clip(np.expm1(r[:, 0]), 0, 1500)
            rows.append(dict(state=k, L=L, payload_med=np.median(pay),
                             iat_p90_ms=1e3 * np.percentile(iat, 90), p_iat_gt_1s=(iat > 1).mean()))
    return pd.DataFrame(rows)


cond = pd.concat([conditional_summary(L) for L in (5, 50, 500, 5000)])
cond.to_csv(art / 'flow_length_response.csv', index=False)
print(cond.pivot(index='state', columns='L', values=['p_iat_gt_1s', 'payload_med']).round(4))